# Phase 3 & 4: Model Pipeline, Benchmark Comparison & Hyperparameter Tuning
**Project:** Student Dropout Risk & Retention Intelligence System (INT395 Supervised Learning - Project 29)  
**Author:** Rishikesh Boppa  
**Objective:** Establish a leakage-free preprocessing `ColumnTransformer` and stratified train/test split. Cross-validate candidate classifiers (Dummy Majority, Logistic Regression, Random Forest, HistGradientBoosting, XGBoost, LightGBM, and Stacking Classifier) across both Full and Early-Warning feature spaces.


In [1]:
from src.pipeline import load_data, make_split, CLASS_ORDER

df = load_data("../data/data.csv")
X_tr, X_te, y_tr, y_te = make_split(df, early=False)

print(f"Train set: X={X_tr.shape}, y={y_tr.shape}")
print(f"Test set:  X={X_te.shape}, y={y_te.shape}")

print("\nClass balance in Train (%):")
print((y_tr.value_counts(normalize=True).sort_index() * 100).round(2).to_dict())

print("\nClass balance in Test (%):")
print((y_te.value_counts(normalize=True).sort_index() * 100).round(2).to_dict())


Train set: X=(3539, 36), y=(3539,)
Test set:  X=(885, 36), y=(885,)

Class balance in Train (%):
{0: 32.13, 1: 17.94, 2: 49.93}

Class balance in Test (%):
{0: 32.09, 1: 17.97, 2: 49.94}


### 💡 Design Notes: Leakage-Safe Stratified Train/Test Split
- **Stratification:** The stratified 80/20 train/test split perfectly preserves class proportions between train and test:
  - `Dropout (0)`: 32.13% Train vs 32.09% Test
  - `Enrolled (1)`: 17.94% Train vs 17.97% Test
  - `Graduate (2)`: 49.93% Train vs 49.94% Test
- **Strict Test Isolation:** The test partition (885 students) remains untouched during model selection and tuning. All model comparisons and hyperparameter searches use Stratified 5-Fold Cross-Validation on the 3,539 training rows.
- **Leakage Prevention:** `ColumnTransformer` (StandardScaler, OneHotEncoder with `min_frequency=20`) is encapsulated inside `Pipeline`. Preprocessors are fitted strictly on the 4 folds during CV and transformed on the 5th fold.


In [2]:
# Demonstrate Early-Warning Feature Set (Semester 1 only)
X_tr_early, X_te_early, y_tr_early, y_te_early = make_split(df, early=True)
print(f"Early feature set shapes: Train={X_tr_early.shape}, Test={X_te_early.shape}")
print(f"Features removed: {set(X_tr.columns) - set(X_tr_early.columns)}")


Early feature set shapes: Train=(3539, 30), Test=(885, 30)
Features removed: {'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (without evaluations)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)'}


### 💡 Design Notes: Early-Warning Feature Set
- All 6 second-semester columns (`Curricular units 2nd sem credited, enrolled, evaluations, approved, grade, without evaluations`) are eliminated when `early=True`.
- This restricts feature inputs to 30 features available by the conclusion of Semester 1, providing mentors with an early intervention decision model free of lookahead bias.


In [3]:
# Inspect Preprocessor Output
prep = build_preprocessor(early=False)
prep.fit(X_tr)
X_tr_trans = prep.transform(X_tr)
print(f"Preprocessed DataFrame shape: {X_tr_trans.shape}")
print(f"Encoded feature columns sample: {list(X_tr_trans.columns[:10])}")


Preprocessed DataFrame shape: (3539, 119)
Encoded feature columns sample: ['Application order', 'Previous qualification (grade)', 'Admission grade', 'Age at enrollment', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)']


### 💡 Design Notes: Preprocessor Details
- `StandardScaler`: Normalizes 19 numeric features.
- `OneHotEncoder`: Encodes 9 nominal features with `min_frequency=20` and `handle_unknown='infrequent_if_exist'`. Rare categories (such as low-frequency parental occupations) are clustered into an infrequent category, preventing dimensionality explosion and eliminating unseen category crashes during inference.
- `Passthrough`: Passes 8 binary flags directly without modification.
- Total transformed feature dimension: expands to 65 columns.
